# 04. Diễn giải kết quả gom cụm và kết luận

**Học phần:** Khai thác dữ liệu — Nhóm 12

Notebook này mô tả chân dung từng cụm, ghép cụm giữa các năm để so sánh xu hướng, hậu kiểm bằng
điểm hạnh phúc và nêu kết luận, hạn chế của đề tài.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.feature_engineering import scale_features, apply_pca, FEATURE_COLUMNS
from src.models.clustering import run_kmeans
from src.visualization.plots import plot_clusters_2d, plot_cluster_radar

figures_dir = project_root / 'reports' / 'figures'

## 1. Chân dung các cụm
1. **Trực quan hóa không gian giảm chiều PCA 2D/3D** để xem ranh giới giữa các cụm.
2. **Lập bảng Profile và vẽ biểu đồ Radar Chart** so sánh trung bình 6 yếu tố giữa các cụm.
3. **Đặt tên có ý nghĩa kinh tế - xã hội cho từng cụm**.

In [ ]:
df = pd.read_csv(project_root / 'data' / 'interim' / 'happiness_merged.csv')
df_2019 = df[df['year'] == 2019].dropna(subset=FEATURE_COLUMNS).copy().reset_index(drop=True)
X_scaled, scaler = scale_features(df_2019, method='standard')
labels, kmeans_model = run_kmeans(X_scaled, n_clusters=3)
df_2019['cluster'] = labels

# Trực quan hóa không gian PCA 2D
X_pca, pca = apply_pca(X_scaled, n_components=2)
plot_clusters_2d(X_pca, labels, title='Phân Bố Các Cụm Trên Không Gian Giảm Chiều 2D PCA', save_path=str(figures_dir / 'pca_clusters_2d.png'))
plt.show()

In [ ]:
# Lập bảng giá trị trung bình 6 yếu tố cho từng cụm
cluster_means = df_2019.groupby('cluster')[FEATURE_COLUMNS].mean()
display(cluster_means)

# Vẽ biểu đồ Radar Chart
plot_cluster_radar(cluster_means, FEATURE_COLUMNS, save_path=str(figures_dir / 'radar_chart_clusters.png'))
plt.show()

### Hậu kiểm (Post-clustering Validation) với Happiness Score và Happiness Rank
Kiểm tra xem các cụm được phân chia dựa trên 6 yếu tố độc lập có thực sự tách biệt rõ ràng về mức điểm hạnh phúc (`happiness_score`) hay không.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(x='cluster', y='happiness_score', data=df_2019, palette='Set2')
plt.title('Hậu Kiểm: Phân Bố Happiness Score Giữa Các Cụm (ANOVA test)', fontsize=14)
plt.savefig(figures_dir / 'validation_happiness_score_by_cluster.png', dpi=300, bbox_inches='tight')
plt.show()

# Thống kê điểm hạnh phúc trung bình từng cụm
print(df_2019.groupby('cluster')['happiness_score'].agg(['count', 'mean', 'std', 'min', 'max']))

In [ ]:
# Xuất dữ liệu hoàn chỉnh gán cụm phục vụ ứng dụng Streamlit
processed_path = project_root / 'data' / 'processed' / 'final_clustered_happiness.csv'
df_2019.to_csv(processed_path, index=False)
print('Đã xuất dữ liệu hoàn chỉnh sang:', processed_path)

## 3. Kết luận, hạn chế và hướng phát triển

### 1. Kết luận chính:
- Thuật toán K-Means với $k=3$ đã tách biệt thành công 156 quốc gia thành 3 nhóm rõ rệt: Nhóm phát triển toàn diện (High Happiness), Nhóm trung bình/đang phát triển (Medium Happiness), và Nhóm có hoàn cảnh khó khăn (Low Happiness / High Vulnerability).
- Hậu kiểm cho thấy điểm `happiness_score` giữa 3 cụm có sự phân hóa cực kỳ sâu sắc dù không hề đưa biến này vào quá trình gom cụm.

### 2. Hạn chế của bộ dữ liệu và phương pháp:
- **Hạn chế dữ liệu:** Dữ liệu khảo sát Cantril Ladder dựa trên cảm nhận chủ quan của người dân; một số quốc gia thiếu dữ liệu một vài năm hoặc có sự thay đổi về câu hỏi khảo sát.
- **Hạn chế phương pháp:** K-Means giả định các cụm có dạng cầu lồi và phương sai bằng nhau, có thể chưa phản ánh hết các cụm có hình dạng phức tạp; DBSCAN nhạy cảm với tham số mật độ.

### 3. Hướng phát triển và mở rộng:
- Phân tích gom cụm theo chuỗi thời gian (Time-series Clustering) từ 2015 đến 2019 để theo dõi sự chuyển dịch cụm của từng quốc gia.
- Tích hợp thêm các chỉ số vĩ mô khác (chỉ số Gini về bất bình đẳng thu nhập, phát thải CO2, vị trí địa lý).